# Fine-tune CodeT5+ on ACRR's validated transformation dataset

**Run this in Google Colab** (`Runtime > Change runtime type > T4 GPU` for reasonable speed — it will run on CPU too, just much slower).

This is Day 3 of the CodeT5+ plan (see `CODET5_PLAN.md` / `SESSION_REPORT.md` in the repo root for the full record of Days 1–2). It:

1. Clones the ACRR repo and regenerates the validated dataset from source (`app/ml/t5/pairs.py`) — the dataset file itself is gitignored, deterministic regeneration from committed templates is the source of truth, not a manually uploaded artifact.
2. Fine-tunes `Salesforce/codet5p-220m-py` on the 150 training pairs.
3. Generates refactorings for the 50 **held-out** pairs — two entire transformation families the model never saw during training.
4. Runs those generations through the SAME differential-testing harness from Day 1 (`app/verify/differential.py`) to check whether they actually behave the same as the original code.
5. Reports the plan's go/no-go gate automatically for the two criteria that can be automated here; the third (beats a baseline) needs a manual comparison against ACRR's live AST rules / Groq path, which this notebook has no access to.

**The gate, from the plan:**
1. ≥95% of outputs pass differential verification
2. Succeeds on ≥1 held-out transformation family it never trained on
3. Beats at least one baseline at something real (manual, see the end of this notebook)

If gates 1–2 fail, the plan's own instruction applies: document it as a negative result, the same way the CodeBERT attempt was — the verification harness (Day 1) already stands on its own regardless of what happens here.

## 1. Setup

In [ ]:
!git clone https://github.com/Ziyad-Firos/Efficode-ACRR.git
%cd Efficode-ACRR/backend
!pip install -q -r requirements.txt
!pip install -q transformers accelerate datasets

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU -- this will run, but slowly. Runtime > Change runtime type > T4 GPU.")

## 2. Regenerate the validated dataset

`pairs_validated.jsonl` is gitignored (matches the project's convention of excluding generated/regenerable artifacts, same as the trained RandomForest `.pkl`). Regenerating it here, in Colab's Linux environment, is also a real second confirmation that the verification harness (built and tested on Windows) works correctly cross-platform — the whole reason it was built to avoid POSIX-only APIs in the first place.

In [ ]:
import subprocess

result = subprocess.run(
    ["python", "-m", "app.ml.t5.build_dataset"],
    capture_output=True, text=True,
)
print(result.stdout[-4000:])
if result.returncode != 0:
    print(result.stderr[-4000:])
    raise RuntimeError(
        "Dataset validation failed. Per the plan: this means a bug in a "
        "template, never a reason to skip the check. Do not proceed."
    )

In [ ]:
import json
from pathlib import Path

pairs_path = Path("app/ml/t5/pairs_validated.jsonl")
rows = [json.loads(line) for line in pairs_path.read_text(encoding="utf-8").strip().split("\n")]

train_rows = [r for r in rows if r["split"] == "train"]
holdout_rows = [r for r in rows if r["split"] == "holdout"]
holdout_families = sorted(set(r["family"] for r in holdout_rows))

print(f"train: {len(train_rows)}   holdout: {len(holdout_rows)}")
print(f"holdout families (the fine-tune will NEVER see these during training): {holdout_families}")

## 3. Prepare training examples

In [ ]:
MODEL_NAME = "Salesforce/codet5p-220m-py"
MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 512

def format_input(before_code: str) -> str:
    # Prompt format kept explicit per the plan, so the model learns intent
    # ("optimize this"), not just style.
    return f"optimize:\n{before_code}"

train_examples = [
    {"input": format_input(r["before"]), "target": r["after"]}
    for r in train_rows
]

print(train_examples[0]["input"])
print("---")
print(train_examples[0]["target"])

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    model_inputs = tokenizer(
        batch["input"], max_length=MAX_SOURCE_LENGTH, truncation=True, padding="max_length",
    )
    labels = tokenizer(
        text_target=batch["target"], max_length=MAX_TARGET_LENGTH, truncation=True, padding="max_length",
    )
    # -100 tells the loss function to ignore padding tokens in the target.
    labels["input_ids"] = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in seq]
        for seq in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

raw_dataset = Dataset.from_list(train_examples)
tokenized = raw_dataset.map(tokenize_fn, batched=True, remove_columns=raw_dataset.column_names)
# A small internal split purely for loss monitoring during training -- NOT
# the real holdout (that's the two families excluded entirely above, used
# only after training finishes, in section 5).
tokenized = tokenized.train_test_split(test_size=0.1, seed=42)
print(tokenized)

## 4. Fine-tune

Starting hyperparameters from the plan: `lr 5e-5`, effective batch size 32 (8 × 4 grad-accum), a handful of epochs, fp16 on GPU. 150 training examples is a small dataset by fine-tuning standards — watch the eval loss; if it's still dropping at the last epoch, this needs more epochs or more data (see `pairs.py`'s docstring on scaling to the full family/instance count), not necessarily a sign of failure at this scale.

In [ ]:
from transformers import (
    T5ForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)

model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

OUTPUT_DIR = "./codet5p-220m-acrr-finetuned"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    fp16=torch.cuda.is_available(),
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    predict_with_generate=True,
    report_to=[],
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

## 5. Generate refactorings for the held-out families

These 50 examples, across 2 whole transformation families, were never in the training data. This is the actual test of whether fine-tuning taught the model something general, or just memorised 150 examples' shapes.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

def generate_refactor(before_code: str, num_beams: int = 4) -> str:
    inputs = tokenizer(
        format_input(before_code), return_tensors="pt",
        max_length=MAX_SOURCE_LENGTH, truncation=True,
    ).to(device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_length=MAX_TARGET_LENGTH, num_beams=num_beams)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

generations = []
for r in holdout_rows:
    generated = generate_refactor(r["before"])
    generations.append({**r, "generated": generated})

print(f"Generated {len(generations)} candidate refactorings for held-out families.")
print()
print("Example (first holdout case):")
print("BEFORE:\n", generations[0]["before"])
print("GENERATED:\n", generations[0]["generated"])

## 6. Verify every generation with Day 1's harness

Same `verify_equivalent` used throughout this project, on real generated output, not a hand-picked example.

In [ ]:
import sys
sys.path.insert(0, ".")
from app.verify.differential import verify_equivalent

verified_count = 0
generalizes_to_families = set()
results = []

for g in generations:
    try:
        eq = verify_equivalent(
            g["before"], g["generated"], g["function_name"],
            trials=10, timeout=3.0,
            custom_values=g.get('custom_values') or None,
        )
        passed = eq.equivalent
        note = eq.note
    except Exception as exc:
        passed = False
        note = f"verification crashed: {exc}"

    if passed:
        verified_count += 1
        generalizes_to_families.add(g["family"])

    results.append({
        "family": g["family"], "function_name": g["function_name"],
        "passed": passed, "note": note,
    })

pass_rate = verified_count / len(generations) if generations else 0.0
print(f"Verified: {verified_count}/{len(generations)}  ({pass_rate:.1%})")
print(f"Held-out families with >=1 passing generation: {sorted(generalizes_to_families)}")

## 7. Go / no-go verdict

In [ ]:
GATE_1_THRESHOLD = 0.95

gate_1 = pass_rate >= GATE_1_THRESHOLD
gate_2 = len(generalizes_to_families) >= 1

print("=" * 64)
print("GO / NO-GO GATE")
print("=" * 64)
print(f"1. >=95% pass differential verification : {'PASS' if gate_1 else 'FAIL'}  ({pass_rate:.1%})")
print(f"2. Succeeds on >=1 held-out family        : {'PASS' if gate_2 else 'FAIL'}  ({len(generalizes_to_families)}/{len(holdout_families)} families)")
print(f"3. Beats a baseline at something real     : MANUAL -- see the note below")
print()
if gate_1 and gate_2:
    print(">> Gates 1-2 PASS. Do the manual gate-3 comparison below before shipping anything.")
else:
    print(">> At least one automatic gate FAILED.")
    print(">> Per the plan: ship the verification harness (already done, Day 1 -- it")
    print("   already upgraded the live Groq path regardless of this result) and stop")
    print("   here. Document this as a negative result the same way the CodeBERT")
    print("   attempt was documented (CODEBERT_DUPLICATE_DETECTION_PLAN.md) -- a clear")
    print("   record of what was tried and why, not a deleted experiment.")
print()
print("Per-family breakdown:")
from collections import defaultdict
by_family = defaultdict(list)
for r in results:
    by_family[r["family"]].append(r["passed"])
for family, passes in sorted(by_family.items()):
    print(f"  {family:30} {sum(passes)}/{len(passes)} passed")

### Gate 3 (manual): does this beat a baseline at something real?

This notebook has no access to ACRR's live AST rules or Groq path (those run inside the deployed backend, not here). To check gate 3 by hand:

1. Take a few of the held-out `before` snippets above.
2. Run them through ACRR's `/refactor` endpoint (rule-based + Groq) and compare:
   - Did the AST rules catch this transformation at all? (Likely no — these are exactly the patterns too semantic for a rule to safely auto-apply.)
   - Did Groq produce an equally good rewrite? If so, on what basis does the fine-tuned 220M model add anything beyond running locally/offline (see `CODET5_PLAN.md` §1 for the honest value-proposition ledger written before this was attempted)?
3. Record the comparison in `SESSION_REPORT.md` or a new dated note, the same way every other result this project has produced was recorded — including if the honest answer is "Groq alone was just as good, and simpler."

## 8. Save artifacts

In [ ]:
import json

with open("gate_results.json", "w", encoding="utf-8") as f:
    json.dump({
        "pass_rate": pass_rate,
        "gate_1_pass": gate_1,
        "gate_2_pass": gate_2,
        "generalizes_to_families": sorted(generalizes_to_families),
        "holdout_families": holdout_families,
        "results": results,
    }, f, indent=2)

print("Wrote gate_results.json -- download it and the model directory below to keep a record.")

try:
    from google.colab import files
    !zip -rq codet5p-220m-acrr-finetuned.zip codet5p-220m-acrr-finetuned
    files.download("codet5p-220m-acrr-finetuned.zip")
    files.download("gate_results.json")
except ImportError:
    print("Not running in Colab -- artifacts are in the current directory, download manually.")